# 11 · Soil Moisture Retrieval — Coherence Inversion & Closure Phase SMI

Implements retrieval algorithms from De Zan et al. (2014) and Zheng & Fattahi (2026).


In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from navasar.inversion import calibration_coherence_curve, invert_moisture_from_coherence, calibrate_closure_transfer, insar_smi_timeseries
from navasar.coherence import interferometric_coherence
from navasar.closure import closure_phase_timeseries
from navasar.slc import simulate_slc
plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
RNG = np.random.default_rng(42)
BASE_KW = dict(n_pixels=300, n_layers=150, max_depth=0.25, sand=0.51, clay=0.13)


## 1. Coherence Forward Model & Inversion


In [ ]:
mv1 = 0.10
mv_grid, coh_grid, phase_grid = calibration_coherence_curve(mv1, freq_ghz=1.4)
mv2_true = np.linspace(0.04, 0.40, 12)
coh_obs = []
for m2 in mv2_true:
    gamma = interferometric_coherence(mv1, m2, freq_ghz=1.4)
    noise = RNG.normal(0, 0.01)
    coh_obs.append(np.clip(np.abs(gamma) + noise, 0.05, 0.999))
mv2_retrieved = [invert_moisture_from_coherence(c, mv1, freq_ghz=1.4, observable='magnitude')[0] for c in coh_obs]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(mv_grid, coh_grid, 'b-', lw=2, label=f'Forward Model (mv1 = {mv1})')
ax1.plot(mv2_true, coh_obs, 'ro', label='Observations')
ax1.set_xlabel('Slave Moisture mv2 [m3/m3]')
ax1.set_ylabel('Coherence Magnitude |gamma|')
ax1.set_title('(a) Coherence Forward Calibration Curve')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax2.plot([0, 0.45], [0, 0.45], 'k--', label='1:1 Line')
ax2.plot(mv2_true, mv2_retrieved, 'bs', ms=7, label='Retrieved Moisture')
rmse = np.sqrt(np.mean((np.array(mv2_retrieved) - mv2_true)**2))
ax2.set_xlabel('True Moisture mv2 [m3/m3]')
ax2.set_ylabel('Retrieved Moisture mv2 [m3/m3]')
ax2.set_title(f'(b) Inversion Accuracy (RMSE = {rmse:.3f} m3/m3)')
ax2.grid(True, alpha=0.3)
ax2.legend()
plt.tight_layout()
plt.savefig('../examples/fig13b_coherence_inversion.png', dpi=150, bbox_inches='tight')
plt.show()


## 2. Closure Phase Calibration: Transfer Function


In [ ]:
mv_base = 0.05
anomalies, cp_steps, t_coeff = calibrate_closure_transfer(mv_base=mv_base, anomaly_amps=np.linspace(0.03, 0.22, 5), freq_ghz=1.4)
plt.figure(figsize=(8, 5))
plt.plot(anomalies, cp_steps, 'bo', ms=8, label='Simulated Steps')
plt.plot(anomalies, t_coeff * np.array(anomalies), 'r-', lw=2, label=f'Linear Fit: T_coeff = {t_coeff:.1f} deg/(m3/m3)')
plt.xlabel('Moisture Anomaly [m3/m3]')
plt.ylabel('Closure Phase Step [deg]')
plt.title('Fig. 13c — Closure Phase Linear Calibration (1.4 GHz)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('../examples/fig13c_closure_calibration.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. InSAR-SMI Time-Series Retrieval Pipeline


In [ ]:
T = 20
t_arr = np.arange(T)
mv_series = np.full(T, 0.06)
mv_series[6:10] = 0.18
slc_stk = simulate_slc(mv_series, freq_ghz=1.4, rng=np.random.default_rng(123), **BASE_KW)
cp_series, _, _ = closure_phase_timeseries(slc_stk)
smi = insar_smi_timeseries(np.rad2deg(cp_series), T_coeff=t_coeff)
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
ax1.plot(t_arr, mv_series, 'k-o', ms=4)
ax1.set_ylabel('mv [m3/m3]')
ax1.set_title('(a) True Soil Moisture Time Series')
ax1.grid(True, alpha=0.3)
ax2.plot(t_arr, np.rad2deg(cp_series), 'g-^', ms=4)
ax2.set_ylabel('CP [deg]')
ax2.set_title('(b) Closure Phase Time Series')
ax2.grid(True, alpha=0.3)
ax3.plot(t_arr, mv_series - 0.06, 'k--', lw=1.5, label='True Anomaly')
ax3.plot(t_arr, smi, 'b-s', ms=4, label='Retrieved SMI')
ax3.set_xlabel('Acquisition Index t')
ax3.set_ylabel('Anomaly [m3/m3]')
ax3.set_title('(c) Retrieved InSAR Soil Moisture Index')
ax3.grid(True, alpha=0.3)
ax3.legend()
plt.tight_layout()
plt.savefig('../examples/fig13d_smi_retrieval.png', dpi=150, bbox_inches='tight')
plt.show()
